# HalluciGuard on Kaggle: blackboard vs. baselines on PopQA + FActScore bios

**Notebook settings:** Accelerator = **GPU T4 x2**, Internet = **on**. No API keys are needed:
the generator is Qwen2.5-7B-Instruct and every agent is an open model served locally with Ollama.

Order matters because the GPUs are shared:
1. **Stage 1**: Qwen generates the answers (uses both T4s, then exits and frees them).
2. **Stage 2**: Ollama serves Llama-3.1-8B / Mistral-7B / Gemma-2-9B; the blackboard and the baselines mitigate the answers.
3. **Stage 3**: scoring + comparison tables, zipped to `/kaggle/working/results.zip`.

Every step checkpoints to `runs/`; if the session dies, rerun the cells and they resume.

In [ ]:
# 0. code + dependencies
%cd /kaggle/working
!rm -rf HalluciGuard && git clone -q -b feature/real-blackboard https://github.com/Yaswanth8390/HalluciGuard.git
%cd /kaggle/working/HalluciGuard
!pip install -q -r requirements-kaggle.txt
!git log --oneline -3

In [ ]:
# 0b. offline tests (fake models, a few minutes). Optional, but catches environment problems early.
!python -m pytest tests -q
!python eval_selftest.py 2>&1 | grep -E "^ok|passed|Error|Traceback"

## Settings
`N_POPQA` is split equally over PopQA's popularity terciles (head / torso / tail). Start small (e.g. 60 / 10)
to check everything end to end, then rerun with the full sizes in a fresh `runs/` folder.

In [ ]:
N_POPQA = 300          # PopQA questions (multiple of 3)
N_BIO = 50             # FActScore biographies (max 500)
BUDGET_QA = 8          # LLM calls per PopQA question for blackboard / CoVe / CRAG
BUDGET_BIO = 24        # LLM calls per biography
WORKERS = 4            # parallel items (matches OLLAMA_NUM_PARALLEL)
PROBE = ""             # optional: path to a trained probe .joblib (e.g. /kaggle/input/<dataset>/probe.joblib)
probe_arg = f"--probe {PROBE}" if PROBE else ""

## 1. Data: PopQA (+ Wikipedia KB) and FActScore biographies

In [ ]:
!python prepare_eval_data.py --dataset popqa --n {N_POPQA} --out eval_data/popqa.jsonl --wiki
!python prepare_factscore.py --n {N_BIO} --out eval_data/bio.jsonl
!wc -l eval_data/popqa.jsonl eval_data/popqa_kb.jsonl eval_data/bio.jsonl eval_data/bio_kb.jsonl

## 2. Stage 1: Qwen2.5-7B generates the answers (GPU)
PopQA uses the repo's short-answer system prompt (the model may answer `Unknown`). Without `PROBE` every
answer is mitigated (`--threshold 0` later), so the mitigation results do not depend on the probe.

In [ ]:
!python eval_pipeline.py --dataset eval_data/popqa.jsonl --generate_only --max_new_tokens 24 --batch_size 16 {probe_arg} --out_dir runs/popqa_gen
!python eval_bio.py --dataset eval_data/bio.jsonl --generate_only --batch_size 4 {probe_arg} --out_dir runs/bio
!nvidia-smi --query-gpu=memory.used --format=csv

## 3. Stage 2: start the open agent models (Ollama) and map them to roles
Heterogeneous agents: verification and answering on **Llama-3.1-8B**, planning/writing on **Mistral-7B**,
the conflict resolver on a different family than the verifier, and a separate **FActScore judge (Gemma-2-9B)**
that is never an agent. None of them is Qwen, so they do not share the generator's errors.

In [ ]:
!bash scripts/kaggle_serve_agents.sh

In [ ]:
import os
AGENT_ENV = {
    "LLM_BASE_URL": "http://localhost:11434/v1", "LLM_API_KEY": "ollama",
    "GROQ_MODEL": "llama3.1:8b",            # default for any role not listed below
    "LLM_JSON_MODE": "1",                   # ask the server for JSON objects (small models need it)
    "LLM_TIMEOUT": "300", "LLM_MAX_TOKENS": "1024",
    "LLM_VERIFIER_MODEL": "llama3.1:8b", "LLM_HYPOTHESIS_MODEL": "llama3.1:8b", "LLM_ANSWER_MODEL": "llama3.1:8b",
    "LLM_DECOMPOSER_MODEL": "mistral:7b", "LLM_REFORMULATOR_MODEL": "mistral:7b", "LLM_COMPOSER_MODEL": "mistral:7b",
    "LLM_CORRECTOR_MODEL": "mistral:7b", "LLM_EXTRACTOR_MODEL": "mistral:7b", "LLM_ABSTENTION_MODEL": "mistral:7b",
    "LLM_CONFLICT_MODEL": "mistral:7b",     # resolver: different family than the verifier
    "LLM_BASELINE_MODEL": "llama3.1:8b",    # CoVe / CRAG use one model, as in their papers
    "LLM_JUDGE_MODEL": "gemma2:9b",         # FActScore judge, never an agent
}
os.environ.update(AGENT_ENV)
COMMON = (f"--answers_file runs/popqa_gen/generations.jsonl --kb_file eval_data/popqa_kb.jsonl --threshold 0 "
          f"--isolate_memory --no_llm_abstention --workers {WORKERS} --retries 3 --fallback_on_error")

### 3a. Live smoke test (10 questions)
Check the summary and a couple of traces before the long runs: are verdicts sensible, does JSON parse
(few `failed:` notes in the traces), are LLM calls within budget?

In [ ]:
!python eval_pipeline.py {COMMON} --limit 10 --system blackboard --max_llm_calls {BUDGET_QA} --out_dir runs/smoke
!head -c 3000 runs/smoke/traces.jsonl

## 4. PopQA: all systems on the same answers
`pipeline` = the original fixed orchestrator, `cove` = Chain-of-Verification, `crag` = Corrective RAG,
`blackboard` = the new control loop. `--llm_only_control` (once) also grades Llama answering on its own.

In [ ]:
for system in ["blackboard", "pipeline", "cove", "crag"]:
    extra = "--llm_only_control" if system == "blackboard" else ""
    !python eval_pipeline.py {COMMON} --system {system} --max_llm_calls {BUDGET_QA} {extra} --out_dir runs/popqa/{system}

In [ ]:
!python compare_systems.py --runs blackboard=runs/popqa/blackboard pipeline=runs/popqa/pipeline cove=runs/popqa/cove crag=runs/popqa/crag --reference pipeline --out runs/popqa/comparison.md
!cat runs/popqa/blackboard/summary.md

## 5. Ablations (blackboard only, same budget)
Which knowledge sources carry the effect? Each run switches one off.

In [ ]:
for ks in ["HypothesisGenerator", "QueryReformulator", "ConflictResolver"]:
    !python eval_pipeline.py {COMMON} --system blackboard --max_llm_calls {BUDGET_QA} --disable_ks {ks} --out_dir runs/popqa/bb_minus_{ks}
!python compare_systems.py --runs blackboard=runs/popqa/blackboard minus_hypgen=runs/popqa/bb_minus_HypothesisGenerator minus_reform=runs/popqa/bb_minus_QueryReformulator minus_conflict=runs/popqa/bb_minus_ConflictResolver --reference blackboard --out runs/popqa/ablations.md

In [ ]:
# single-model ablation: every agent role on Llama-3.1-8B (does heterogeneity help?)
single = {k: "llama3.1:8b" for k in AGENT_ENV if k.startswith("LLM_") and k.endswith("_MODEL") and k != "LLM_JUDGE_MODEL"}
saved = {k: os.environ[k] for k in single}
os.environ.update(single)
!python eval_pipeline.py {COMMON} --system blackboard --max_llm_calls {BUDGET_QA} --out_dir runs/popqa/bb_single_llama
os.environ.update(saved)
!python compare_systems.py --runs hetero=runs/popqa/blackboard single_llama=runs/popqa/bb_single_llama --reference single_llama --out runs/popqa/heterogeneity.md

### Optional: knowledge-base coverage
Only half of the items have their gold passage in the KB: covered items should get corrected, uncovered ones
should abstain instead of hallucinating.

In [ ]:
!python eval_pipeline.py --answers_file runs/popqa_gen/generations.jsonl --kb_coverage 0.5 --threshold 0 --isolate_memory --no_llm_abstention --workers {WORKERS} --retries 3 --fallback_on_error --system blackboard --max_llm_calls {BUDGET_QA} --out_dir runs/popqa/bb_cov50

## 6. FActScore biographies (long-form): mitigation + scoring

In [ ]:
!python eval_bio.py --answers_file runs/bio/generations.jsonl --kb eval_data/bio_kb.jsonl --systems blackboard,pipeline,cove,crag --max_llm_calls {BUDGET_BIO} --workers {WORKERS} --out_dir runs/bio

## 7. Results

In [ ]:
!cat runs/popqa/comparison.md; echo; cat runs/popqa/ablations.md; echo; cat runs/bio/summary.md
!cd /kaggle/working/HalluciGuard && zip -qr /kaggle/working/results.zip runs eval_data/popqa.jsonl eval_data/bio.jsonl && ls -lh /kaggle/working/results.zip